# Offline verification with the missing Requests test plugin

Empty/reference/edit-only controls are repeated with pytest-httpbin installed offline from a separately prepared wheel bundle. Official verifier, test targets and test files are unchanged. These are repaired-local-environment controls, not competition leaderboard scores.


In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

In [ ]:
import importlib.metadata as md, json, os, subprocess, sys
from pathlib import Path
matches=list(Path('/kaggle/input').rglob('dependency-manifest.json'))
assert len(matches)==1, f'Expected one dependency manifest: {matches}'
extra_root=matches[0].parent/'request-test-wheels'
manifest=json.loads(matches[0].read_text())
import hashlib
for item in manifest:
    assert hashlib.sha256((extra_root/item['name']).read_bytes()).hexdigest()==item['sha256']
pytest_before=md.version('pytest')
constraints=Path('/tmp/verification-pytest-constraint.txt')
constraints.write_text('pytest=='+pytest_before+'\n')
subprocess.run([sys.executable,'-m','pip','install','--no-index','--find-links',str(extra_root),
               '--constraint',str(constraints),'pytest-httpbin==2.1.0'],check=True)
assert md.version('pytest')==pytest_before, 'Verification must preserve official pytest version'
plugins=[{'name':ep.name,'value':ep.value} for ep in md.entry_points(group='pytest11')]
assert any('httpbin' in ep['value'] for ep in plugins),plugins
Path('/kaggle/working/environment-repair.json').write_text(json.dumps({
    'pytest_version':pytest_before,'pytest_httpbin_version':md.version('pytest-httpbin'),
    'plugins':plugins,'wheel_manifest':manifest,
    'scope':'Offline local verification dependency repair; official verifier and test files unchanged'},indent=2))
print('Test plugin installed offline; unchanged pytest:',pytest_before)


In [ ]:
import asyncio, concurrent.futures, hashlib, json, time
from pathlib import Path
from adk_submission import ModelRegistry
from swegemma.config import EvalConfig
from swegemma.models import load_tasks
from swegemma.deduplication import resolve_task_snapshot_paths
from swegemma.sandbox import SubprocessManager
from swegemma.harness.verification import verify_task
from swegemma.results import append_task_result

DATA_DIR=Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR=Path('/kaggle/working')
TASKS_PATH=DATA_DIR/'tasks.jsonl'
SPLIT=json.loads('{"schema_version": 1, "seed": "main-track-20261108-v1", "task_file_sha256": "e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6", "selection_fields": ["repo", "instance_id"], "development": ["rich_3942", "rich_4079", "rich_3882", "httpx_3672", "fastapi_13713", "fastapi_15661", "fastapi_14463", "requests_7433", "requests_7205", "requests_7502"], "confirmation": ["rich_3772", "rich_3278", "rich_3535", "rich_3930", "rich_3782", "rich_3676", "rich_3105", "rich_3064", "rich_3180", "rich_3468", "rich_3521", "rich_3472", "rich_3130", "rich_4076", "rich_3480", "rich_3944", "rich_3486", "rich_3938", "rich_4077", "rich_3470", "rich_3935", "rich_3067", "rich_3469", "rich_3454", "rich_3718", "rich_4070", "rich_2943", "rich_3471", "rich_3777", "rich_4075", "rich_3934", "rich_3953", "rich_3061", "rich_3905", "rich_3052", "rich_2725", "rich_3043", "rich_3006", "rich_4006", "rich_3063", "rich_3518", "rich_3675", "rich_3894", "rich_3296", "rich_3506", "fastapi_14616", "fastapi_14258", "fastapi_15800", "fastapi_9753", "fastapi_14297", "fastapi_13786", "fastapi_14448", "fastapi_15023", "fastapi_14262", "fastapi_14479", "fastapi_14605", "fastapi_14303", "fastapi_14371", "fastapi_14356", "fastapi_12942", "fastapi_5624", "fastapi_14246", "fastapi_14186", "fastapi_14077", "fastapi_15763", "fastapi_15589", "fastapi_13207", "fastapi_14360", "fastapi_13537", "fastapi_14430", "fastapi_14492", "fastapi_14419", "fastapi_15785", "fastapi_15280", "fastapi_14512", "fastapi_14349", "fastapi_14794", "fastapi_15030", "fastapi_14583", "fastapi_9425", "fastapi_14962", "fastapi_14482", "fastapi_9555", "fastapi_5077", "fastapi_14485", "fastapi_14978", "fastapi_11355", "fastapi_14851", "fastapi_15588", "fastapi_14791", "fastapi_14487", "fastapi_14873", "fastapi_14099", "fastapi_14964", "fastapi_14458", "fastapi_14301", "fastapi_14953", "fastapi_11194", "fastapi_14986", "fastapi_14361", "fastapi_14609", "fastapi_14266", "fastapi_14459", "fastapi_14306", "fastapi_15745", "fastapi_14786", "fastapi_14372", "fastapi_14455", "fastapi_13920", "requests_6589", "requests_7505", "requests_7315", "requests_7427", "requests_6757", "requests_7309", "requests_7328", "requests_6629", "requests_6644", "requests_6592"], "smoke_task_id": "requests_7433", "repo_counts": {"Textualize/rich": 48, "encode/httpx": 1, "fastapi/fastapi": 67, "psf/requests": 13}, "confirmation_is_external_benchmark": false, "small_repo_note": "Repositories with <=3 tasks have no confirmation tasks."}')
EDIT_ONLY_PATCH='--- a/src/requests/models.py\n+++ b/src/requests/models.py\n@@ -596,7 +596,9 @@\n             if not isinstance(body, bytes):\n                 body = body.encode("utf-8")\n \n-        if isinstance(data, Iterable) and not isinstance(\n+        if (\n+            isinstance(data, Iterable) or hasattr(data, "__iter__")\n+        ) and not isinstance(\n             data, (str, bytes, list, tuple, Mapping)\n         ):\n             try:\n'
assert hashlib.sha256(TASKS_PATH.read_bytes()).hexdigest()==SPLIT['task_file_sha256']
task=next(task for task in load_tasks(TASKS_PATH) if task.instance_id==SPLIT['smoke_task_id'])
assert task.instance_id in SPLIT['development']
assert task.patch.strip() and task.test_patch.strip()
snapshot,base_snapshot,snapshot_patch=resolve_task_snapshot_paths(DATA_DIR/'snapshots',task.instance_id,task.repo)
assert snapshot.exists()

def run_sync(coroutine):
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(lambda:asyncio.run(coroutine)).result()

# Reference labels are used only in this host-side verification control.
# No model or repair agent is instantiated in this notebook.
for arm, patch in [('empty_control',''),('reference_control',task.patch),('baseline_edit_only',EDIT_ONLY_PATCH)]:
    output=WORKING_DIR/'results'/arm
    config=EvalConfig(tasks_path=TASKS_PATH,snapshots_dir=DATA_DIR/'snapshots',
        results_dir=output,submission_dir=DATA_DIR/'sample_submission',models=ModelRegistry(),
        sandbox='subprocess',wheels_dir=DATA_DIR/'wheels',timeout_seconds=60,
        verification_timeout_seconds=300,display_mode='quiet')
    manager=SubprocessManager(timeout_seconds=60)
    result=run_sync(verify_task(manager,config,task,snapshot,base_snapshot_path=base_snapshot,
        patch_path=snapshot_patch,agent_patch=patch,start_time=time.perf_counter()))
    append_task_result(result,output,[result])
    record={'arm':arm,'task_id':task.instance_id,'resolved':result.resolved,
            'test_exit_code':result.test_exit_code,'error':result.error,
            'duration_seconds':result.duration_seconds,'purpose':'verification_environment_control',
            'candidate_is_original_agent_patch':False}
    with (WORKING_DIR/'verification_results.jsonl').open('a') as out:out.write(json.dumps(record)+chr(10))
    print(json.dumps(record))
    print((result.test_output or '')[-5000:])
